# SHW: Sihyun skeleton + v9 · train100

This notebook runs the single-file `clarify/algorithms/shw_classifier.py` and saves
its actual generation, grading and summary outputs in the cells below.

The algorithm keeps Sihyun's original detectors and sampling placeholders. Property
judgments fill the existing slots. Three methods add the v9 sequence: freeze the
specification → generate/review tests → repair code using real test failures.

The fixed sample is the same 100 training development problems used previously:
HumanEval/MBPP × original/contradictory variants, 25 per group, seed 20261030.
Its task IDs and prompt hashes are embedded below; no extra benchmark files are needed.
This is a development sample, and variant names are not contradiction ground truth.

Select a Python 3.13 kernel with the project dependencies, `ipykernel` and `nbformat`.
Configure credentials in the environment or an ignored `.env`. Start Docker before
execution. Generation uses the configured language/decision models; grading uses
saved code and the SDK evaluator. Save this notebook after execution to share results.


In [1]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import Markdown, display
from dotenv import load_dotenv
import copy
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "clarify/algorithms/shw_classifier.py").is_file()
            and (p / "data/splits/train.txt").is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from clarify.data import preprocess_benchmark

RUN_ID = "shw-sihyun-v9-20261103"
MODEL = os.environ.get("ECC_OPENAI_MODEL", "openai/gpt-4.1-mini")
DECISION_MODEL = os.environ.get("ECC_DECISION_MODEL", "openai/gpt-6-luna")
MAX_TURNS = 1
WORKERS = 8
MAX_TEST_VERSIONS = 3
MAX_CODE_ATTEMPTS = 4
RUN_GENERATION = True
RUN_GRADING = True
OUTPUT_DIR = ROOT / ".local" / RUN_ID
ALGORITHM = ROOT / "clarify/algorithms/shw_classifier.py"
print("Repository:", ROOT)
print("Models:", MODEL, DECISION_MODEL)
print("Output:", OUTPUT_DIR)


/Users/kwonhyukwon/.codex/worktrees/sihyun-review/ECC/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository: /Users/kwonhyukwon/developer/research-bera/ECC
Models: openai/gpt-4.1-mini openai/gpt-6-luna
Output: /Users/kwonhyukwon/developer/research-bera/ECC/.local/shw-sihyun-v9-20261103


## Frozen problem selection

Each entry is `(source dataset, task ID, expected prompt SHA-256)`. These identify
existing repository data, including the reference material used only by the SDK
simulator/evaluator. Participant code receives only `prompt` and `entry_point`.


In [2]:
SAMPLE = [
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/107', '0635c205473d52d0dcb6681641abe4b97606c0f5dbd6b2881179bd386b9f0d64'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/87', '8de3ecf44a0ece8d4e372c37aa51439548bd77398c179f009834d5b5d1ea34fb'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/33', '612811faaa6ea7c9b210b7d499677c1efbdccf4d5c9100dae9974f2d7609ac99'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/62', 'c01e27585092fc19d56719138e5c3a7a957b600fc3dc6e0f6fb5f220bc223bec'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/75', 'f79fd26f4e9ff99e9ae381b75a16079018508c41352e7f1224b768bc5c9b1ade'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/55', '24e01045b87209e3f363bd1846abb7c0de51d10cd97cb1f0301d1a740c5bd82f'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/156', '84a0896623d6a306393249f57eb69f5d229c02d935c795bbf043e5030e78e415'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/43', 'f20511ba41a7533e3f9593f29edff94e9190de6b789e91155c06a6b40eb83917'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/90', 'c411484e97c83ae8e5869ae414e5687f429deda4f5fb49e367e60c85635fed1e'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/116', 'dc8d425f9133ada84c2a380d6cab8321aba622d443cee4d54bcc98c4859a2289'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/21', 'f5d1f07614da8dffb7e6ede02cdedd73f58405f47aeb516b82318416a4acf0c7'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/100', '429d50357e5a2020cb173089bd82e4813107af7741ef8bca4c2390118ae3cad2'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/143', '85e7c088d3b44a74767bc7408853a709f18dd89c1ef964694103be8ee8dca8e1'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/106', 'f9dee5694b9f6e2c5f39785ca58d27c74624ba052d3010d58eececbe949ce19d'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/149', '442ea867c9d6737dc58fced1d3aaac3c726af1290cbfa41b4b7c737b0d01f8ed'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/42', '8b6d8ac13f22a485fb80312ee1b077ba1cc2653fbff4c36fb7e3d36ca1b8d609'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/111', '24d001dd5f00b4701f06e4017368a6a2d45d7e3789d00313249f0a05a4004c20'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/105', '7dfc8f34546dd7ce8ddd64fcefb1e8069202bee6057858313c3c8803d6065e97'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/162', '3d1a07a0032583310d116f768148924428ec7e61f72a9451dadee884360942fb'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/119', '7d057f437949f329d47d033084ee94a506621185f9063e13a91bbb741dca6216'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/123', 'bfe35061d0207e39a86aaa1c69efffacd4632a1409ab7739b89b264cf467414c'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/59', '67b0503335f6c8c2f212dec159e3b6068863e78ddb1ee5a953505e88b43bf13b'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/47', '261f6d801ea6fa9a1958bde6144b83a6e36ff45abce4d5f2ca1f0ade07931685'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/88', 'eccc8ccc94ea27ceb8d1511b8986c7665564ac9560bab0f38c44e29584ec69a9'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/14', '1fced7e32ef6a751f365fa8aa968cb5db6539387d201afbfa12df297a5a8fa98'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/127', '5d767ddc19678edd7693d23d4409c00875eb243d3bd5203a4749d0bdfd884648'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/137', 'a64051f2757f643cda80ad30999ec15eebdae7c3748c8541b353d8f59e380a36'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/144', '2a0addf544cd38216f359d1e48649ae659b89cd25c4f7fa72060ee8cbb404735'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/89', 'efefc17723390b1692f2f4fdbb26dac3ecc055be739dbaaa2a020f5b18d4999a'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/13', 'a7946546bc3a483bb5f2249ced878d510d6765d3c987a3e56dd878fe5bb60c19'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/93', '856a164439599802d5210e2969c1c5673c84b83b4bdca5db34384d7b10d3d741'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/136', '9dd2a92d4b141565a666931ff044c4c6109db09520ed710c3651e9ae51d80ec8'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/52', 'b8e47fee4b6fffb27f872307ef74803b1e427e22802413851b9f0c61bb05306e'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/7', 'd750c9285c8aede9860b779ebdb3e9ca274137e80be7bee1bddcdbe13bec84a5'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/24', '3bc8bf3f66a3b5e171a358bbd5f33fba1a5d16ea65459198757dcb662dbaaac2'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/31', '6e6a26b011e1a2de922a6afc0ce6176fc04c3c068b7f571712b0269584ef5136'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/28', '6d4e2dc1fa1fac2a4e38aac76669e8c7a576a5dbe3879010f42e61ee75960a43'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/44', '6c3987abb35a3e0bf61eb7bd7e709b0abecef72c47dcb61a6ba84b0d46d760a0'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/140', 'f757e5d21a3d47b21dcd96c7c9f869adfbfe70276370fddb0f4ded8fb1c311f9'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/95', '6325c35e466d6fddcee5421520509b4d670380ed4e5da7edcfab999ea1d7e48d'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/157', 'badc999ecbcb5e6dd1dbb3d1c1b89f9b624e434dd4afebf241d035353b2b0a79'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/99', '53ad185333496f1faa011070d323b24af3e23506e32f52ced0b3c0f9867d2719'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/153', 'c0ecd987e3c84c2b50926b5a4061c8386ec1e6b0704e3ce8630d46ca5a3144ca'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/18', '47f223b2abcd895890f76fed8084b9dec676b98d6b7fff8aaede0fc5ef6c4839'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/84', '8bc15887209916cd222d5610d4ff2ef6f9b86765f3393ee3e391757bf250e660'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/154', '66607b421ed8b5eb91de52ca96f1b071ecc536edf716451650816ae4e7701f64'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/54', '34580b5dedd0034c619b8f76bb976cbcfb6a828f0df6d963f6a545ae94b59531'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/0', 'ccfc6ec967ae4f5866b10849d79f8343ac7f565a5eeb6cb776963a947b3aa062'),
    ('data/humaneval/humaneval_larbi_contradictory.jsonl', 'HumanEval/11', 'e95ef73c273da62b6c490080efc39fc38d6b3c32a1c42b06fdd0ca213767c289'),
    ('data/humaneval/humaneval_original.jsonl', 'HumanEval/67', '287ace00706dbc14460387cbd37396f40de3816fb26bd13182cec8a85fd6dddb'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/770', 'c834d4bc3cfb88c125034f0c70a6bd68fb28456ea2faa7fbb3c2b22b7e5ff962'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/268', '1c25b60d52ce633e6b39f01376ea60b0716b4137e3badb85a6a421b19432a6cd'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/38', '318c9bb306eb635d527a9694155136b19d2bc4da14ced43e21f240c8956100f2'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/306', '8d6f6aa18055daead4f2358b1a868073ef51995fcb0982f2e01dbf4f41ba8a69'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/232', 'bb16d8dd505b8150afe65b65c986a589c8328ccfc9e733e60210bf1c87d5d5ee'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/641', '6d5d037856aa182cb83868487f978f2c211ea9183553921277afd5d1ef340094'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/762', '9445725ab7b6edd43e69938b4d89d31065f52b756f93784fb17a09ba29cd603f'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/382', '70d9db6d5a66e9304f4e0605a1d46fe9b9aa7312ce0a9c5c1dd93d204e5135fd'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/135', '3504f477a4ba41916c019e47e485c76ea2651f58023b5b649012fe5ed07ffbf8'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/589', 'ab2fb6599ea1a28a95505c721f5773701dfde25aa70a4a7e4a3a8d8636034e8e'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/183', '5dbf700a0ba30451d445c8299b64ade7e099371383b1e4d76e7380a6c129edfc'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/736', '6b8c5b3438f008311fc5c76521d5bce85e2d5dd06cc293f9528a0abdb77bb53c'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/577', '5a7b83ecddf5fb168addf0321557e1dfd67794a9d3bd6932ee318143d7aa0a1f'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/645', 'd6d996e0fac7b76353a2e53a0de9cd66e578aff268d917d9dbc4e5d882b167ac'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/144', 'd731623fd09b9946b562601cde2a3e26b7495756cec5aff2e101f7ede0a13a1b'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/772', '1ffc6b0e03d1655a7ce982b6e418337528dd4b2c45d75646fd208b5cafdfad51'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/423', '3e3e0b9b6ec9a7085af154ff57939c53ce945d55716e73bccfdb40e2251d15de'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/600', '51782d0e8cffd203eb4698754cd248e6c78ac9569ca3abcf5c234ed83ef63302'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/958', 'b8252b46dfe9fbf70dc04ac9ce6bef0a5bbf98237f3e2d604753a4ce58566b05'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/328', 'ba76b2dff039d7d19bca95bbc81a0bd291f774c905af6cec25cb1d90751d95aa'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/84', '79ef84e6c34c0edb7446f52eb491bce4aed6fb0240ebf9d5c69726db7e22a6be'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/698', '588ef6d59306ef31169a5aca0ec733ba6473391de97987fcb96e99e28859c5f1'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/628', 'f28a84fc7a46c4266f8702b89774ab9f1cf3a294fe5d6265aba88323d8af53b1'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/461', '20ac8a4e17ead5177b4fdae1dac1930f8b63f2c8d0e52fa1dcb8b2c575b34d19'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/378', 'dd8d480c2e0276507331abfd02ee216572fe00ba0492841c27f1adb393e8be8d'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/941', '63fe0f302e1dace69890da6c535b8c4a7156d7f6732c2dc23e75c9b2856e4623'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/199', 'f3f8eaaf7d74e6cd902da47a9e31a882d012b2e349a2a835b5b227ed603b4d67'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/18', '634adb23c7b23afe0727e395e42c6d3d3dae1059dca1b365fb8242df21ff9ba5'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/611', 'd0df064f924be98e4c01ad768f70c3978eb21412c26de22a82ec2873b8d8ac7c'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/414', '1c3fc9f8b4fbc691a32da92372dda162e3730fa9aa9e5942270551aec48f5e3e'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/308', 'ebe4dbbf072b59c8a50985b908403cf2d2e8721b08c25aee62886d9b4c338cd5'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/409', '57953931dc6e0b10404bda422c258e637fb49b8612a25d30578b2effe022db62'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/753', '9218e37b0e7d2549e784522223dad80118b531c52b50143b42b9da98fdfd59d3'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/311', 'afe295e85d443200b6827ef90101a469657f3e17f8d4eb3c357893bfb15c391f'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/361', 'e75538cb9e746c29a1e533b43f5f855609fcf266a8c1f26a4b4523a2e106ee0d'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/627', '23b3a1bbbb41ac5a1e9cc7a4f763b68d8543968e0e36be7749f0ffb290ea05f4'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/253', 'e4fe7d34d41402f6f7673c049e5ed639ede7d040461c241b607c7f8d326de6cc'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/723', '7934048e9f42cc5ca07589efdb7a4e170774beb4f74cd9542c6cac7b67ca7126'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/763', '1bfc55033b1927d6b5b4fcc4d6fd6e45aa5fcb033b539c9fda0f278249afc22e'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/227', '7ada012cc5598b2d3fa6063c0a3c5c2f9e6ab849c9ff81bd889abf6c301e0666'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/7', '2243872d4f9c4a49093433b0e8dbe543f4985b7d1057697f7341205fce60396c'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/222', '2e1b707c85f6a20607a04d5a49c88ff2e5ab712a2ac052587c512a566afe0935'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/70', '03dce36c91eb9aa5b163928b0c3fb479fbaf2c41e38253e10d4e038176633d78'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/165', '788c9033750f9b02a1c577c92f8ea51bfc3fba52f28a3365b37a0d7c6103a000'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/247', 'bd604df29ff3639fe5aabc0f812905e831ed036c71e282363297619fa4c94e78'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/874', '76c6fc5a5a0f53083ec3ce46fd3e0212a2adac672e1aa41e97fd414e65830037'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/166', '445c3da7af6428a3ec4987ffd873af131cd3c9e42705721b425a6218aac3ef6c'),
    ('data/mbpp/mbpp_original.jsonl', 'Mbpp/90', '356965dc3946b404e671a822ff705a0c6ef787133b531a5e1602e66e28187687'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/936', '3e0cd24dc537abee8d4849b0c10d24aa916728334ca1ce138f8cae33c7d9cff0'),
    ('data/mbpp/mbpp_larbi_contradictory.jsonl', 'Mbpp/373', '6330eca353ad39e2d38a974ea2f4783c3b124bddef90477385c3dddfc214369b'),
]

loaded = {}
rows = []
sample_info = {}
train_ids = set((ROOT / "data/splits/train.txt").read_text().splitlines())
validation_ids = set((ROOT / "data/splits/validation.txt").read_text().splitlines())
for source, task_id, prompt_sha in SAMPLE:
    if source not in loaded:
        raw = [json.loads(line) for line in (ROOT / source).read_text().splitlines() if line.strip()]
        loaded[source] = copy.deepcopy(preprocess_benchmark(raw))
    row = copy.deepcopy(loaded[source][task_id])
    assert hashlib.sha256(row["prompt"].encode()).hexdigest() == prompt_sha, task_id
    assert task_id in train_ids and task_id not in validation_ids, task_id
    rows.append(row)
    dataset = task_id.split("/")[0]
    variant = "original" if source.endswith("_original.jsonl") else "contradictory"
    sample_info[task_id] = (dataset, variant)
assert len(rows) == len({r["task_id"] for r in rows}) == 100
print("Verified 100 distinct training tasks:", dict(Counter(sample_info.values())))


Verified 100 distinct training tasks: {('HumanEval', 'original'): 25, ('HumanEval', 'contradictory'): 25, ('Mbpp', 'contradictory'): 25, ('Mbpp', 'original'): 25}


## Prepare the run

Generated files and logs stay under the existing ignored `.local/` directory.
The run snapshots the one algorithm file and records code/SDK hashes and settings.
Use a new run ID if these change. Completed batches are reused on rerun; partial
or mismatched batch files stop execution rather than being overwritten.


In [3]:
sdk_files = ["generate_responses.py", "evaluate_responses.py", "clarify/env.py",
             "clarify/llm.py", "clarify/runtime.py", "clarify/data.py", "uv.lock"]
settings = {
    "algorithm_sha256": hashlib.sha256(ALGORITHM.read_bytes()).hexdigest(),
    "sdk_sha256": {p: hashlib.sha256((ROOT / p).read_bytes()).hexdigest() for p in sdk_files},
    "sample_sha256": hashlib.sha256(json.dumps(SAMPLE).encode()).hexdigest(),
    "model": MODEL, "decision_model": DECISION_MODEL, "max_turns": MAX_TURNS,
    "max_test_versions": MAX_TEST_VERSIONS, "max_code_attempts": MAX_CODE_ATTEMPTS,
    "max_prompt_budget_per_task": 1.0, "max_clarification_budget_per_task": 1.0,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
state_path = OUTPUT_DIR / "settings.json"
if state_path.exists():
    assert json.loads(state_path.read_text()) == settings, "Settings/source changed: use a new RUN_ID."
    assert hashlib.sha256((OUTPUT_DIR / "algorithm.py").read_bytes()).hexdigest() == settings["algorithm_sha256"]
else:
    assert not any(OUTPUT_DIR.iterdir()), "Nonempty output directory has no settings."
    state_path.write_text(json.dumps(settings, indent=2) + "\n")
    shutil.copyfile(ALGORITHM, OUTPUT_DIR / "algorithm.py")
batches = []
for index in range(10):
    batch = rows[index * 10:(index + 1) * 10]
    path = OUTPUT_DIR / f"batch-{index + 1:02d}-input.jsonl"
    content = "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in batch)
    if path.exists():
        assert path.read_text() == content, "Frozen batch changed."
    else:
        path.write_text(content)
    batches.append((index + 1, path, batch))
print(json.dumps(settings, indent=2))


{
  "algorithm_sha256": "280a9868291852b08b45850a7f995109577cbc5fa049f201fc57db3d7797cb38",
  "sdk_sha256": {
    "generate_responses.py": "93885748cf069c053334f2518c1218f289000eaa2e10e33f0b604936f1a1b2f2",
    "evaluate_responses.py": "960996e5ad752672322e705c62cc1b07941edb4c94359ae7c004931406cd9f47",
    "clarify/env.py": "1cd0126345bd703392bab99b9bdc255a9071ae578b78739bf13816ae903d086e",
    "clarify/llm.py": "34b46e6a0fd54e56ad0419e09043253f86875af146d2de2d2347a762543f5df7",
    "clarify/runtime.py": "8f4029ffba4b611c2520f7d7b21af82105a3d1fb7da1f2d2b89b6c9befaae148",
    "clarify/data.py": "1d64ee6af1842721db3e1a7fc33a76c91eb073b874934475c6e748cc22191926",
    "uv.lock": "45ac05568d2a0eea6eb4423538cfbefd3cc4692ca68a47f74e613cbd235d42e9"
  },
  "sample_sha256": "853f56c14c2333a11280a171c627682f18a8b4473c4e205f4106f157f10b1eff",
  "model": "openai/gpt-4.1-mini",
  "decision_model": "openai/gpt-6-luna",
  "max_turns": 1,
  "max_test_versions": 3,
  "max_code_attempts": 4,
  "max_promp

## Generate the 100 implementations

This uses the existing SDK entry point, with separate subprocesses and log files
for each batch. Logs contain model messages, property pair decisions, delta-debugging
observations, Q/A, frozen specifications, test reviews, execution results and code feedback.
The model's generated tests are separate from the official tests used in the next cell.


In [4]:
def generate_batch(item):
    index, input_path, expected = item
    output = OUTPUT_DIR / f"batch-{index:02d}-output.jsonl"
    log = OUTPUT_DIR / f"batch-{index:02d}-generate.log"
    expected_keys = {(r["task_id"], r["prompt"]) for r in expected}
    if not output.exists():
        if log.exists():
            raise RuntimeError(f"Inspect the earlier interrupted attempt before retrying: {log}")
        command = [sys.executable, "generate_responses.py", str(OUTPUT_DIR / "algorithm.py"),
            "--dataset_path", str(input_path), "--output_path", str(output),
            "--language_model", MODEL, "--clarification_model", MODEL,
            "--decision_model", DECISION_MODEL, "--max_clarification_turns", str(MAX_TURNS),
            "--max_test_versions", str(MAX_TEST_VERSIONS), "--max_code_attempts", str(MAX_CODE_ATTEMPTS),
            "--max_prompt_budget", "1.0", "--max_clarification_budget", "1.0",
            "--max_workers", "1", "--fail_on_exception=False"]
        with log.open("w") as handle:
            process = subprocess.run(command, cwd=ROOT, stdout=handle, stderr=subprocess.STDOUT)
        if process.returncode:
            raise RuntimeError(f"Generation process failed: {log}")
    result = [json.loads(line) for line in output.read_text().splitlines() if line.strip()]
    assert len(result) == len(expected) and {(r["task_id"], r["prompt"]) for r in result} == expected_keys, output
    return index, result

if RUN_GENERATION:
    docker = subprocess.run(["docker", "info"], capture_output=True)
    assert docker.returncode == 0, "Start Docker Desktop."
    generation_started = time.monotonic()
    collected = {}
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = [pool.submit(generate_batch, batch) for batch in batches]
        for future in as_completed(futures):
            index, result = future.result()
            collected[index] = result
            errors = sum(str(r.get("prompt_result", "")).startswith("[EXCEPTION]") for r in result)
            print(f"Batch {index:02d}: {len(result)} completed, {errors} generation errors; "
                  f"total {sum(len(v) for v in collected.values())}/100", flush=True)
    generated = [r for index in sorted(collected) for r in collected[index]]
    (OUTPUT_DIR / "all-output.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in generated))
    print("Generation elapsed seconds:", round(time.monotonic() - generation_started, 1))
else:
    generated = [json.loads(line) for line in (OUTPUT_DIR / "all-output.jsonl").read_text().splitlines()]
print("Generated records:", len(generated))
print("Generation exceptions:", sum(str(r.get("prompt_result", "")).startswith("[EXCEPTION]") for r in generated))


Batch 06: 10 completed, 0 generation errors; total 10/100


Batch 08: 10 completed, 0 generation errors; total 20/100


Batch 07: 10 completed, 0 generation errors; total 30/100


Batch 04: 10 completed, 0 generation errors; total 40/100


Batch 02: 10 completed, 0 generation errors; total 50/100


Batch 03: 10 completed, 0 generation errors; total 60/100


Batch 01: 10 completed, 0 generation errors; total 70/100


Batch 05: 10 completed, 0 generation errors; total 80/100


Batch 09: 10 completed, 0 generation errors; total 90/100


Batch 10: 10 completed, 0 generation errors; total 100/100


Generation elapsed seconds: 1255.3
Generated records: 100
Generation exceptions: 0


## Grade saved implementations

The SDK evaluator runs the official training tests in Docker. This cell performs
no model calls and does not submit or modify a leaderboard. Generation exceptions
are counted separately in the final summary, even when their output fails grading.


In [5]:
results_path = OUTPUT_DIR / "results.jsonl"
if RUN_GRADING and not results_path.exists():
    grade_log = OUTPUT_DIR / "evaluate.log"
    command = [sys.executable, "evaluate_responses.py", "--split", "train",
        "--generation_path", str(OUTPUT_DIR / "all-output.jsonl"),
        "--output_path", str(results_path), "--max_workers", "4"]
    with grade_log.open("w") as handle:
        process = subprocess.run(command, cwd=ROOT, stdout=handle, stderr=subprocess.STDOUT)
    assert process.returncode == 0, f"Inspect grading log: {grade_log}"
results = [json.loads(line) for line in results_path.read_text().splitlines() if line.strip()]
assert len(results) == 100 and {(r["task_id"], r["prompt"]) for r in results} == {(r["task_id"], r["prompt"]) for r in rows}
assert all(type(r.get("success")) is bool for r in results)
print("Graded:", len(results), "/100")


Graded: 100 /100


## Recorded aggregate results

These are one stochastic run on the fixed training development sample. Recorded
SDK costs may omit failed/retried calls. Dataset variants are not detector labels.


In [6]:
passed = sum(r["success"] for r in results)
questions = sum(len(r.get("clarification_history", [])) for r in results)
asked = sum(bool(r.get("clarification_history")) for r in results)
tds = sum(math.log(10) / math.log(10 + len(r.get("clarification_history", [])))
          for r in results if r["success"]) / 100
ndcg = 0.0
for row in results:
    history = row.get("clarification_history", [])
    weights = [math.log(2) / math.log(2 + i) for i in range(len(history))]
    if weights:
        ndcg += sum(w for w, turn in zip(weights, history, strict=True) if turn[2] == "3") / sum(weights) / 100
summary = {
    "total": 100, "passed": passed, "pass_rate": passed / 100, "tds": tds, "sdk_ndcg": ndcg,
    "asked_tasks": asked, "questions": questions,
    "quality3_questions": sum(turn[2] == "3" for r in results for turn in r.get("clarification_history", [])),
    "generation_exceptions": sum(str(r.get("prompt_result", "")).startswith("[EXCEPTION]") for r in generated),
    "recorded_cost": sum(float(r.get(k, 0) or 0) for r in generated for k in ("prompt_cost", "clarification_cost")),
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
display(summary)
lines = ["| Dataset | Variant | Passed | Asked |", "|---|---|---:|---:|"]
for group in sorted(set(sample_info.values())):
    selected = [r for r in results if sample_info[r["task_id"]] == group]
    lines.append(f"| {group[0]} | {group[1]} | {sum(r['success'] for r in selected)}/{len(selected)} | "
                 f"{sum(bool(r.get('clarification_history')) for r in selected)} |")
display(Markdown("\n".join(lines)))


{'total': 100,
 'passed': 52,
 'pass_rate': 0.52,
 'tds': 0.504101027115651,
 'sdk_ndcg': 0.8600000000000005,
 'asked_tasks': 86,
 'questions': 86,
 'quality3_questions': 86,
 'generation_exceptions': 0,
 'recorded_cost': 1.8718473999999998}

| Dataset | Variant | Passed | Asked |
|---|---|---:|---:|
| HumanEval | contradictory | 6/25 | 25 |
| HumanEval | original | 19/25 | 22 |
| Mbpp | contradictory | 10/25 | 25 |
| Mbpp | original | 17/25 | 14 |

## Per-problem results and detailed inspection

The table shows all 100 outcomes. Select a task below to inspect its final code,
Q/A, official grading result and the corresponding raw generation log. No new
model calls are made when inspecting saved results.


In [7]:
lines = ["| Task | Variant | Passed | Questions | Recorded cost |", "|---|---|---|---:|---:|"]
for row in results:
    cost = sum(float(row.get(k, 0) or 0) for k in ("prompt_cost", "clarification_cost"))
    lines.append(f"| {row['task_id']} | {sample_info[row['task_id']][1]} | {row['success']} | "
                 f"{len(row.get('clarification_history', []))} | ${cost:.4f} |")
display(Markdown("\n".join(lines)))
TASK_ID = "HumanEval/107"
detail = next(r for r in results if r["task_id"] == TASK_ID)
display(detail)
batch_number = next(index for index, _, batch in batches if any(r["task_id"] == TASK_ID for r in batch))
print("Full generation log:", OUTPUT_DIR / f"batch-{batch_number:02d}-generate.log")
print("Full grading log:", OUTPUT_DIR / "evaluate.log")


| Task | Variant | Passed | Questions | Recorded cost |
|---|---|---|---:|---:|
| HumanEval/107 | original | True | 1 | $0.0255 |
| HumanEval/87 | original | False | 1 | $0.0283 |
| HumanEval/33 | contradictory | True | 1 | $0.0174 |
| HumanEval/62 | contradictory | True | 1 | $0.0281 |
| HumanEval/75 | contradictory | False | 1 | $0.0243 |
| HumanEval/55 | contradictory | False | 1 | $0.0245 |
| HumanEval/156 | contradictory | True | 1 | $0.0197 |
| HumanEval/43 | original | True | 1 | $0.0117 |
| HumanEval/90 | original | True | 1 | $0.0165 |
| HumanEval/116 | original | False | 1 | $0.0211 |
| HumanEval/21 | original | True | 1 | $0.0058 |
| HumanEval/100 | original | True | 1 | $0.0170 |
| HumanEval/143 | original | True | 1 | $0.0226 |
| HumanEval/106 | original | True | 1 | $0.0163 |
| HumanEval/149 | contradictory | False | 1 | $0.0241 |
| HumanEval/42 | original | True | 0 | $0.0197 |
| HumanEval/111 | contradictory | True | 1 | $0.0245 |
| HumanEval/105 | contradictory | False | 1 | $0.0190 |
| HumanEval/162 | contradictory | True | 1 | $0.0203 |
| HumanEval/119 | original | True | 1 | $0.0406 |
| HumanEval/123 | contradictory | False | 1 | $0.0268 |
| HumanEval/59 | contradictory | False | 1 | $0.0236 |
| HumanEval/47 | contradictory | False | 1 | $0.0269 |
| HumanEval/88 | contradictory | False | 1 | $0.0267 |
| HumanEval/14 | contradictory | False | 1 | $0.0135 |
| HumanEval/127 | contradictory | False | 1 | $0.0222 |
| HumanEval/137 | contradictory | True | 1 | $0.0242 |
| HumanEval/144 | original | True | 1 | $0.0180 |
| HumanEval/89 | original | False | 1 | $0.0185 |
| HumanEval/13 | original | True | 1 | $0.0209 |
| HumanEval/93 | original | False | 1 | $0.0199 |
| HumanEval/136 | original | True | 0 | $0.0203 |
| HumanEval/52 | original | True | 1 | $0.0147 |
| HumanEval/7 | contradictory | False | 1 | $0.0221 |
| HumanEval/24 | original | True | 0 | $0.0240 |
| HumanEval/31 | contradictory | False | 1 | $0.0271 |
| HumanEval/28 | contradictory | False | 1 | $0.0199 |
| HumanEval/44 | original | True | 1 | $0.0240 |
| HumanEval/140 | original | False | 1 | $0.0192 |
| HumanEval/95 | contradictory | False | 1 | $0.0130 |
| HumanEval/157 | original | True | 1 | $0.0272 |
| HumanEval/99 | original | False | 1 | $0.0243 |
| HumanEval/153 | original | True | 1 | $0.0231 |
| HumanEval/18 | contradictory | False | 1 | $0.0131 |
| HumanEval/84 | contradictory | False | 1 | $0.0210 |
| HumanEval/154 | original | True | 1 | $0.0240 |
| HumanEval/54 | contradictory | False | 1 | $0.0283 |
| HumanEval/0 | contradictory | False | 1 | $0.0301 |
| HumanEval/11 | contradictory | False | 1 | $0.0219 |
| HumanEval/67 | original | True | 1 | $0.0211 |
| Mbpp/770 | contradictory | True | 1 | $0.0046 |
| Mbpp/268 | original | True | 0 | $0.0105 |
| Mbpp/38 | contradictory | True | 1 | $0.0051 |
| Mbpp/306 | original | False | 1 | $0.0216 |
| Mbpp/232 | original | True | 1 | $0.0187 |
| Mbpp/641 | original | True | 0 | $0.0153 |
| Mbpp/762 | original | True | 0 | $0.0058 |
| Mbpp/382 | contradictory | True | 1 | $0.0140 |
| Mbpp/135 | original | True | 0 | $0.0072 |
| Mbpp/589 | original | False | 0 | $0.0144 |
| Mbpp/183 | contradictory | False | 1 | $0.0206 |
| Mbpp/736 | contradictory | False | 1 | $0.0085 |
| Mbpp/577 | original | False | 1 | $0.0072 |
| Mbpp/645 | contradictory | False | 1 | $0.0171 |
| Mbpp/144 | contradictory | False | 1 | $0.0196 |
| Mbpp/772 | original | True | 1 | $0.0209 |
| Mbpp/423 | contradictory | False | 1 | $0.0445 |
| Mbpp/600 | original | True | 1 | $0.0106 |
| Mbpp/958 | contradictory | True | 1 | $0.0057 |
| Mbpp/328 | contradictory | False | 1 | $0.0181 |
| Mbpp/84 | original | True | 0 | $0.0161 |
| Mbpp/698 | contradictory | False | 1 | $0.0201 |
| Mbpp/628 | contradictory | False | 1 | $0.0174 |
| Mbpp/461 | contradictory | False | 1 | $0.0050 |
| Mbpp/378 | contradictory | False | 1 | $0.0110 |
| Mbpp/941 | contradictory | True | 1 | $0.0196 |
| Mbpp/199 | contradictory | True | 1 | $0.0183 |
| Mbpp/18 | original | True | 1 | $0.0132 |
| Mbpp/611 | original | True | 0 | $0.0064 |
| Mbpp/414 | original | False | 1 | $0.0070 |
| Mbpp/308 | contradictory | False | 1 | $0.0237 |
| Mbpp/409 | original | True | 0 | $0.0181 |
| Mbpp/753 | contradictory | False | 1 | $0.0149 |
| Mbpp/311 | original | False | 0 | $0.0397 |
| Mbpp/361 | contradictory | True | 1 | $0.0040 |
| Mbpp/627 | contradictory | False | 1 | $0.0181 |
| Mbpp/253 | original | True | 1 | $0.0092 |
| Mbpp/723 | contradictory | True | 1 | $0.0155 |
| Mbpp/763 | original | False | 1 | $0.0195 |
| Mbpp/227 | original | True | 0 | $0.0061 |
| Mbpp/7 | original | False | 1 | $0.0253 |
| Mbpp/222 | original | True | 1 | $0.0174 |
| Mbpp/70 | original | False | 1 | $0.0080 |
| Mbpp/165 | original | True | 1 | $0.0167 |
| Mbpp/247 | contradictory | False | 1 | $0.0236 |
| Mbpp/874 | contradictory | True | 1 | $0.0174 |
| Mbpp/166 | original | True | 0 | $0.0232 |
| Mbpp/90 | original | True | 1 | $0.0148 |
| Mbpp/936 | contradictory | False | 1 | $0.0215 |
| Mbpp/373 | contradictory | True | 1 | $0.0251 |

{'task_id': 'HumanEval/107',
 'prompt': '\ndef even_odd_palindrome(n):\n    """\n    Given a positive integer n, return a tuple that has the number of even and odd\n    integer palindromes that fall within the range(1, n), inclusive.\n\n    Example 1:\n\n        Input: 3\n        Output: (1, 2)\n        Explanation:\n        Integer palindrome are 1, 2, 3. one of them is even, and two of them are odd.\n\n    Example 2:\n\n        Input: 12\n        Output: (4, 6)\n        Explanation:\n        Integer palindrome are 1, 2, 3, 4, 5, 6, 7, 8, 9, 11. four of them are even, and 6 of them are odd.\n\n    Note:\n        1. 1 <= n <= 10^3\n        2. returned tuple has the number of even and odd integer palindromes respectively.\n    """\n',
 'algorithm': 'SHWClassifier',
 'model': 'openai/gpt-4.1-mini',
 'algorithm_path': '/Users/kwonhyukwon/developer/research-bera/ECC/.local/shw-sihyun-v9-20261103/algorithm.py',
 'prompt_result': '```python\ndef even_odd_palindrome(n):\n    even_count = 0\n 

Full generation log: /Users/kwonhyukwon/developer/research-bera/ECC/.local/shw-sihyun-v9-20261103/batch-01-generate.log
Full grading log: /Users/kwonhyukwon/developer/research-bera/ECC/.local/shw-sihyun-v9-20261103/evaluate.log
